# Parte 6. Investigación de las herramientas

El objetivo de este cuaderno es documentar conceptualmente las herramientas que utilizaremos para el procesamiento, agregación y reducción de datos masivos.

---

## 1. Datashader

* **¿Para qué sirve?**  
  Es una librería de visualización de Python diseñada para procesar y renderizar conjuntos de datos masivos (millones o miles de millones de puntos) de forma rápida y sin incurrir en el problema de sobrelapamiento (*overplotting*).

* **¿Qué tipo de datos maneja?**  
  Maneja datos puntuales 2D, series temporales, mallas espaciales, grafos/redes y datos tabulares masivos.

* **¿Qué significa agregación?**  
  Es el proceso de proyectar o agrupar puntos de datos continuos en una rejilla discreta (*canvas* o píxeles), acumulando métricas estadísticas (como conteo, suma o promedio) por cada celda de la grilla.

* **¿Cómo utiliza bins/rejillas?**  
  Divide el espacio visual en una matriz de celdas o *bins* (correspondientes a los píxeles de la imagen a generar). Asigna cada punto de datos a su bin correspondiente y calcula la densidad agregada.

* **¿Cómo puede representar millones de puntos?**  
  En lugar de intentar dibujar individualmente millones de geometría/puntos vectoriales en la pantalla (lo cual colapsaría la memoria/navegador), Datashader primero calcula la densidad acumulada en la rejilla de píxeles y luego aplica un mapa de colores (*colormap*). La complejidad computacional escala con la resolución de la pantalla, no con el número total de puntos.

---

## 2. Vaex

* **¿Para qué sirve?**  
  Es una biblioteca de Python orientada al análisis, exploración y visualización de DataFrames tabulares masivos que superan la capacidad de la memoria RAM disponible (*out-of-core DataFrames*).

* **¿Cómo trabaja con datasets grandes?**  
  Utiliza **Memory Mapping** (`mmap`) y **evaluación perezosa** (*lazy evaluation*). Esto le permite leer directamente archivos en disco (HDF5, Arrow, Parquet) sin cargarlos por completo en la memoria RAM, procesando datos a velocidades de miles de millones de filas por segundo.

* **¿Qué operaciones de agregación permite?**  
  Permite realizar estadísticas descriptivas sobre grillas N-dimensionales (conteo, media, desviación estándar, mínimo, máximo, cuantiles e histogramas de alta velocidad en 1D y 2D).

* **¿Cómo trabaja con columnas?**  
  Trabaja mediante expresiones virtuales de columnas (*virtual columns*). Las transformaciones matemáticas sobre columnas no crean nuevas copias de datos en RAM, sino que se almacenan como fórmulas y se evalúan únicamente cuando es necesario renderizar o exportar.

---

## 3. DuckDB

* **¿Qué es?**  
  Es un sistema de gestión de bases de datos relacional analítico en memoria (OLAP) e incrustado (*embedded*). Se le conoce popularmente como el "SQLite para análisis de datos".

* **¿Cómo permite consultar datos tabulares?**  
  Permite realizar consultas relacionales directamente sobre archivos CSV, Parquet, JSON, Arrow o DataFrames de Pandas y Polars en disco o memoria, sin necesidad de realizar una importación previa a una base de datos propia.

* **¿Cómo se pueden hacer agregaciones y cuantiles?**  
  Cuenta con un motor de ejecución vectorial multihilo altamente optimizado que ejecuta funciones de agregación SQL avanzadas como `COUNT`, `AVG`, `SUM`, `PERCENTILE_CONT`, `APPROX_PERCENTILE` y `QUANTILE_CONT`.

* **¿Cómo se relaciona con SQL?**  
  Soporta el dialecto estándar SQL y proporciona extensiones analíticas avanzadas. Permite integrar sintaxis SQL directamente en scripts de Python o R.

---

## 4. Polars

* **¿Qué es?**  
  Es una biblioteca de manipulación de DataFrames extremadamente rápida escrita en Rust, diseñada para procesamiento en paralelo y uso eficiente de memoria.

* **¿Cómo representa DataFrames?**  
  Utiliza la especificación **Apache Arrow** en memoria. Almacena los datos en un formato columnar contiguo que optimiza el acceso a la caché de la CPU y permite vectorización SIMD.

* **¿Qué son las operaciones lazy/eager?**  
  * **Eager (Eterna/Inmediata):** Ejecución directa paso a paso sobre los datos (estilo Pandas).  
  * **Lazy (Perezosa):** Construye un plan de consulta (*Query Plan*) estructurado como un Grafo Acíclico Dirigido (DAG). Polars optimiza automáticamente este plan (filtrado temprano, proyección de columnas) antes de ejecutar el cómputo final con `.collect()`.

* **¿Qué operaciones de agrupación y transformación ofrece?**  
  Proporciona expresiones potentes de agrupación mediante `.group_by().agg()`, operaciones por ventana (`over()`), transformaciones vectoriales, reestructuración (*pivot/melt*) y filtrado optimizado.

---

## 5. Scikit-Learn (KBinsDiscretizer)

* **¿Qué es KBinsDiscretizer?**  
  Es una clase transformadora de Scikit-Learn que convierte variables continuas numéricas en intervalos o *bins* discretos (codificados como valores ordinales o categóricos/one-hot).

* **¿Qué significa discretizar una variable?**  
  Consiste en dividir un rango de valores continuos en un conjunto finito de intervalos acotados (*bins*), convirtiendo un atributo numérico continuo en un atributo categórico u ordinal para reducir la variabilidad o adaptar los datos a modelos específicos.

* **Estrategia `uniform`:**  
  Todos los intervalos (*bins*) tienen el mismo ancho o amplitud constante a lo largo del rango de la variable: $bin\_width = (max - min) / k$.

* **Estrategia `quantile`:**  
  Todos los intervalos tienen aproximadamente la misma cantidad de puntos u observaciones. La anchura del bin cambia según la distribución de datos (basado en percentiles).

* **Estrategia `kmeans`:**  
  Aplica un algoritmo de agrupamiento K-Means unidimensional sobre los datos. Los bordes de los *bins* se definen en función de los centroides calculados por el algoritmo.

---

# Parte 7. Tabla de investigación

| Herramienta      |                    Propósito                     | Técnica que estudiare | Tipo de reducción     |
| :--- | :---      | :---                                                                                       |:--- |
| **Datashader**   | Agregación y representación de grandes volúmenes | Binning / rejillas           |   Agregación   |
| **Vaex**         | Procesamiento de datasets grandes                | Binning / agregaciones       |  Agregación    |
| **DuckDB**       | Consultas analíticas                             | GROUP BY / cuantiles         |  Agregación    |
| **Polars**       | Procesamiento tabular                            | Agregación / cambio de tipos | Simplificación |
| **Scikit-Learn** | Machine Learning y transformación                | KBinsDiscretizer             | Discretización |

---

# Parte 8. Definir qué vamos a medir

Para evaluar de manera sistemática cada herramienta de reducción y agregación, se registrarán y analizarán las siguientes cuatro dimensiones de métricas:

### 1. Rendimiento
* **Tiempo de lectura:** Tiempo transcurrido durante el ingreso/carga de datos.
* **Tiempo de procesamiento:** Tiempo requerido para aplicar la transformación o agregación.
* **Tiempo total:** Duración completa del flujo desde la lectura hasta el resultado final.

### 2. Memoria
* **Memoria inicial:** Consumo de RAM antes de iniciar la operación.
* **Memoria durante procesamiento:** Pico máximo de memoria RAM utilizada.
* **Memoria final:** Consumo de RAM una vez liberados los recursos temporales.

### 3. Reducción
* **Número de filas original vs. resultante:** Reducción en la cardinalidad del conjunto de datos.
* **Tamaño original vs. tamaño reducido:** Comparativa en megabytes (MB) o gigabytes (GB).
* **Porcentaje de reducción (%):** Porcentaje de compresión o simplificación logrado.

### 4. Conservación de información
* **Estadísticos descriptivos:** Media, Desviación Estándar, Mínimo y Máximo.
* **Distribución:** Comparativa de cuantiles e histogramas entre el dataset original y el reducido.

---

> **Nota:**  
> Para garantizar una comparación homogénea, todas las herramientas serán evaluadas utilizando el mismo conjunto de datos y métricas equivalentes de tiempo, consumo de memoria, reducción de tamaño y conservación de características estadísticas.